# MATH840 — Lab 3, worked example

**One pass through the whole assignment, start to finish, on a series nobody was assigned.**

Queensland clothing retailing, monthly, 1982–2018. It is drawn from `aus_retail`, which is
**not** in the pool your own series comes from — so this is a demonstration of the method,
not a template you can copy answers out of.

The section headings are the ones your submission must have. Watch what goes in each, and
in particular how much of the work is *sentences* rather than code.

## 0. Setup

The same given cells as in the template: nothing here is marked.

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from coreforecast.scalers import boxcox, boxcox_lambda

BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

SEASON = 12
FREQ = "MS"
mine = {"label": "Queensland clothing retailing", "freq": FREQ, "season": SEASON}

retail = pd.read_csv(f"{BASE}/aus_retail.csv", parse_dates=["Month"])
raw = (retail[(retail["State"] == "Queensland")
              & (retail["Industry"] == "Clothing retailing")]
       .sort_values("Month")[["Month", "Turnover"]]
       .rename(columns={"Month": "ds", "Turnover": "y"})
       .reset_index(drop=True))

In [ ]:
s = raw.groupby("ds", as_index=False)["y"].mean() if raw["ds"].duplicated().any() else raw.copy()
grid = pd.date_range(s["ds"].min(), s["ds"].max(), freq=mine["freq"])
s = s.set_index("ds").reindex(grid).rename_axis("ds")
s["y"] = s["y"].interpolate(limit_area="inside")
s = s.dropna().reset_index()

print(f"{len(s)} observations, {s['ds'].min().date()} to {s['ds'].max().date()}, "
      f"{len(s) / mine['season']:.1f} seasonal cycles")

For this series the preparation changes nothing — no duplicates, no gaps, no zeros. What
last week's plots already showed: a strong trend that slows after about 2008, and a December
peak whose size grows with the level. That second observation is where this lab starts.

## 1. Data preparation

*1 point.* **Does the variance need stabilising?** The December swing grows with the level of
the series. That is exactly the situation an additive decomposition handles badly, so estimate
a Box-Cox $\lambda$ and look at what it fixes.

In [ ]:
y = s["y"].to_numpy(float)
lam = boxcox_lambda(y, method="loglik", season_length=SEASON)
work = boxcox(y, lam)

first, last = y[:60], y[-60:]
first_w, last_w = work[:60], work[-60:]
print(f"lambda = {lam:.3f}")
print(f"  raw:    sd of first 5 years = {first.std():7.2f}, last 5 years = {last.std():7.2f}"
      f"  -> x{last.std() / first.std():.1f}")
print(f"  boxcox: sd of first 5 years = {first_w.std():7.2f}, last 5 years = {last_w.std():7.2f}"
      f"  -> x{last_w.std() / first_w.std():.1f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(s["ds"], y, linewidth=0.9, color="#e64173"); axes[0].set_title("original")
axes[1].plot(s["ds"], work, linewidth=0.9, color="#20B2AA")
axes[1].set_title(f"Box-Cox transformed, lambda = {lam:.2f}")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Decision: work transformed.** The spread of the last five years is 6.5 times the spread
of the first five in the original series; after the transformation it is 2.4 times. That is
a large improvement and it makes the additive decompositions below defensible.

Note what the number also says: $\lambda = 0.45$ is roughly a square root, and it does
**not** fully stabilise the variance — a residual growth of 2.4× remains. So this is a
compromise, not a fix. Saying that out loud is worth more than pretending the problem is gone,
and Section 2 will show where the leftover growth ends up.

## 2. EDA and visualisation

*4 points — one per subsection,* all on the transformed series.

### 2.1 Classical decomposition

In [ ]:
classical = seasonal_decompose(work, period=SEASON, model="additive")

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, (name, comp) in zip(axes, [("observed", classical.observed),
                                   ("trend", classical.trend),
                                   ("seasonal", classical.seasonal),
                                   ("remainder", classical.resid)]):
    ax.plot(s["ds"], comp, linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("Classical decomposition")
plt.tight_layout()
plt.show()

resid_classical = pd.Series(classical.resid, index=s["ds"])
trend_classical = pd.Series(classical.trend)
print("trend missing: first", int(trend_classical.head(SEASON).isna().sum()),
      "| last", int(trend_classical.tail(SEASON).isna().sum()), "months")
print(f"seasonal swing: {classical.seasonal[:SEASON].max() - classical.seasonal[:SEASON].min():.2f}, "
      "the same every year")
print("largest remainders:", [str(x.date()) for x in resid_classical.abs().nlargest(4).index])

**Reading the classical components.** The trend rises steeply to about 2008 and grows more
slowly afterwards; six months are missing at each end — the price of a centred 12-month moving
average. The seasonal component is one pattern with a swing of 5.9 units, repeated unchanged
for all 36 years. And the remainder is not noise: its four largest values are all Decembers,
at both ends of the history. That is the fixed seasonal pattern failing where the real one has
moved away from it.

### 2.2 STL

`seasonal=13` smooths each month's seasonal effect across 13 years — wide enough to keep the
pattern steady from one year to the next, narrow enough to let its size change over three
decades. `robust=True` because one-off months should not bend the trend or the season.

In [ ]:
stl = STL(work, period=SEASON, seasonal=13, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for ax, (name, comp) in zip(axes, [("observed", work),
                                   ("trend", stl.trend),
                                   ("seasonal", stl.seasonal),
                                   ("remainder", stl.resid)]):
    ax.plot(s["ds"], comp, linewidth=0.9)
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
fig.suptitle("STL decomposition (seasonal=13, robust)")
plt.tight_layout()
plt.show()

season = pd.Series(stl.seasonal, index=s["ds"])
by_month = season.groupby(season.index.month).mean().round(2)
print("average seasonal effect by month (transformed units):")
print(by_month.to_string())

early, late = season.iloc[:120], season.iloc[-120:]
print(f"\nseasonal amplitude: first 10 years = {early.max() - early.min():.2f}, "
      f"last 10 years = {late.max() - late.min():.2f}")

resid_stl = pd.Series(stl.resid, index=s["ds"])
print("largest remainders:", [str(x.date()) for x in resid_stl.abs().nlargest(4).index])

**Reading the STL components.** The trend matches the classical one — steep to about 2008,
slower after — but runs to both ends of the series.

The seasonal component is dominated by December (+3.7) against a February trough (−2.3) —
the Christmas trade. It is stable in *shape* but not in *size*: the amplitude grows from 4.3
in the first decade to 9.4 in the last, so even after the Box-Cox transform the seasonality is
still becoming more pronounced.

The remainder is small and mostly structureless. Its largest values — March 1999, June and
August 2007, December 2005 — are isolated months, worth a look at the calendar before
assuming they are noise.

### 2.3 Classical versus STL

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(s["ds"], classical.seasonal, linewidth=0.8, color="#e64173", label="classical")
axes[0].plot(s["ds"], stl.seasonal, linewidth=0.8, color="#20B2AA", label="STL")
axes[0].set_title("Seasonal component")
axes[1].plot(s["ds"], classical.resid, linewidth=0.8, color="#e64173", label="classical")
axes[1].plot(s["ds"], stl.resid, linewidth=0.8, color="#20B2AA", label="STL")
axes[1].set_title("Remainder")
for ax in axes:
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

rc = resid_classical.dropna()
missed = (season - pd.Series(classical.seasonal, index=s["ds"]))[rc.index]
print(f"remainder sd: classical {rc.std():.2f} | STL {resid_stl.std():.2f}")
print(f"correlation of the classical remainder with the seasonal change it missed: {rc.corr(missed):.2f}")

**Where they disagree, and which to trust.** On the trend they agree, except that classical
loses six months at each end. On the seasonal component they disagree systematically:
classical fits one swing to every year, while STL lets it grow from about 4 to about 9.
Classical therefore overstates the season early on and understates it late — and that
difference has to go somewhere. It goes into the remainder: the output above shows it about
one and a half times the size of STL's, and correlated at about 0.8 with the seasonal change
classical failed to capture. That is also why its largest values were all Decembers.

**On this series STL is the one to trust** — and the evidence is not a preference but the
classical remainder itself.

### 2.4 Seasonally adjusted series

In [ ]:
adjusted = work - stl.seasonal

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(s["ds"], work, linewidth=0.8, alpha=0.45, label="transformed", color="#e64173")
ax.plot(s["ds"], adjusted, linewidth=1.2, label="seasonally adjusted", color="#20B2AA")
ax.legend()
ax.grid(alpha=0.3)
ax.set_title("Seasonally adjusted series")
plt.show()

**What it is for, and what it hides.** With December and February pulled out, the turning
points are readable: the slowdown after 2008 is obvious here and easy to miss in the raw
series, where every December spike interrupts the eye. What it hides is that the removed
component is the largest single feature of this business — a retailer planning stock cannot
use the adjusted series, because December *is* the year.

## 3. Implementation

*2 points.*

### 3.1 Strength of trend and seasonality

In [ ]:
def strengths(fit) -> tuple[float, float]:
    """F_T and F_S from a fitted STL decomposition."""
    R, T, S = fit.resid, fit.trend, fit.seasonal
    return (max(0.0, 1 - R.var() / (T + R).var()),
            max(0.0, 1 - R.var() / (S + R).var()))


F_T, F_S = strengths(stl)
print(f"F_T = {F_T:.2f}   F_S = {F_S:.2f}")

Both are close to 1: the remainder is tiny next to either component. This series is almost
entirely trend plus season, with very little left over.

### 3.2 Explaining last week's winner

Lab 2 scored the four simple methods on the last year of this series. Recompute that result
first, so the explanation has something to explain.

In [ ]:
def benchmark_forecasts(y, h, m):
    T = len(y)
    return {
        "mean":   np.repeat(y.mean(), h),
        "naive":  np.repeat(y[-1], h),
        "snaive": np.array([y[-m + (i % m)] for i in range(h)]),
        "drift":  y[-1] + np.arange(1, h + 1) * (y[-1] - y[0]) / (T - 1),
    }


# Lab 2 scored the untransformed series on its last year - redo exactly that.
train, test = y[:-12], y[-12:]
rmse = {k: float(np.sqrt(np.mean((test - v) ** 2)))
        for k, v in benchmark_forecasts(train, 12, SEASON).items()}

LAB2_WINNER = min(rmse, key=rmse.get)
for name, value in sorted(rmse.items(), key=lambda kv: kv[1]):
    mark = "  <- winner" if name == LAB2_WINNER else ""
    print(f"  {name:7s} RMSE {value:7.1f}{mark}")

**The explanation.** Seasonal naive won, and not narrowly: RMSE 18.9 against 132–151 for the
other three. The decomposition says why.

$F_S = 0.93$: the seasonal component carries almost all the variance that is not trend, and
STL shows it keeping the same shape for 36 years — December high, February low. Seasonal
naive copies exactly that component forward, so it starts with nearly everything this series
does for free.

The other three have nothing comparable. `mean` ignores both the trend and the seasonality of
a series that is mostly trend and seasonality. `naive` carries the last month forward — and
the last month is December, the peak of the year, so every forecast it makes is biased high.
`drift` extrapolates the average slope of the whole history, and the adjusted series in 2.4
shows that growth slowed after 2008 — so it projects a pace the business no longer keeps.

Note the order: the result came first, last week; the decomposition explains it. That is the
order this lab asks for.

## 4. Code quality and reproducibility

Restart, run all, check that nothing errors and that every number quoted above still comes
out of a cell. That is the whole point.